# Цветовая калибровка цифровых микроскопов
## Явно исправленный расчет

Пройдем путь от исходных спектров и изображений к сравнению охватов и полиномиальной калибровке каждого микроскопа. Выполняйте ячейки последовательно или выберите **Restart Kernel and Run All Cells**.

**Перед началом:** [подготовка данных](../docs/ru/data.md), [оборудование](../docs/ru/hardware.md), [источники и предобработка](../docs/ru/preprocessing.md). Скачайте архивы спектров/справочных данных и изображений; укажите `data_root` в `config.json`. С [английской версией](corrected_en.ipynb) используется один код; меняется только язык пояснений и отображения.

Это официальный маршрут исследования для [статьи о цветовой мере](https://doi.org/10.1007/s11018-022-02014-7) и [статьи о полиномиальной калибровке](https://doi.org/10.1088/1742-6596/2192/1/012010). Входы — сохранившиеся исследовательские файлы. Опубликованное покрытие 99.4% не удается точно получить из доступного набора.

### Что исправлено

При необходимости согласуются длины волн, маскируется неположительное пропускание, сохраняется масштаб OD стекол, нормируются масштабы RGB/XYZ, спектральные Lab рассчитываются напрямую, осветитель исключается из обучения, стабилизируются входы полиномов; используются стандартное среднее ΔE76 и ошибки отложенных полей. Модель без перекрестных членов следует аддитивной формуле статьи. [Все изменения и оставшиеся допущения](../docs/ru/corrections.md) описаны отдельно; кодирование sRGB камеры остается допущением. Это новый анализ тех же данных, а не замена исторического результата.

## 1. Конфигурация и краткая проверка входов

Вспомогательная функция проверяет папки полей и необходимые входы. Таблица кратко показывает найденные файлы; технические проверки вынесены из расчетных ячеек.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "preprocessing").is_dir())
sys.path.insert(0, str(root))
from preprocessing.common import load_config
from analysis import figures, reporting
config = load_config()
PROFILE, LANGUAGE = "corrected", "ru"
figures.setup()
reporting.inventory(config, LANGUAGE)

## 2. Наблюдатель и освещение

Копируем архивные таблицы без изменения сеток. Для охватов используются **D50 и физиологически обоснованный наблюдатель 2° (`2_2006`)**. Для калибровки далее — измеренный осветитель и `10_2006`. CIE 1931/D65 нужен только для цветного фона диаграммы. [Источники и обозначения](../docs/ru/observers.md).

In [ ]:
from preprocessing import observers
observers.main(config)
figures.plot_reference_functions(config, LANGUAGE)

## 3. Чувствительности камеры из исходного графика

В исследовании график Sony ICX274AQ вручную разделили на маски каналов. Повторяем выделение центральной линии и перевод пикселей в длины волн по сохраненным маскам. Кривые описывают **относительную чувствительность сенсора**, не измеренные RGB полей. [Источник, ручной этап и привязка осей](../docs/ru/camera.md).

In [ ]:
from preprocessing import camera
camera.main(config)
figures.plot_camera(config, LANGUAGE)

## 4. Спектры красителей

18 спектров оптической плотности предоставила **Дарья Добролюбова**, бывшая коллега автора и аспирантка МГТУ им. Н. Э. Баумана. Протокол измерений не записан. Переводим векторы длина волны/OD в удобные CSV и смотрим примеры. Пропускание рассчитывается как **T = 10^(−OD)** без нормировки OD каждого красителя. [Происхождение и формат](../docs/ru/dyes.md).

In [ ]:
from preprocessing import dyes
dyes.main(config)
figures.plot_dyes(config, LANGUAGE)

## 5. Другие цветовые меры

Сначала извлекаем девять кривых пропускания из **Bautista и соавт. (2014), рисунок 5a**. Выходной шаг 0.1 нм — сетка интерполяции, не разрешение исходного измерения. [Оцифровка](../docs/ru/bautista.md).

In [ ]:
from preprocessing import bautista
bautista.main(config)
figures.plot_bautista(config, LANGUAGE)

Читаем остальные меры в **исходных форматах**, затем переводим в XYZ. ColorChecker — Lab/D50; Kodak — XYZ в масштабе 0–100; IFRAO — Adobe RGB/D65 в масштабе 0–255; Korzynska — восемь Lab с унаследованным допущением осветителя A. Таблица сохраняет эти различия. Это не взаимозаменяемые спектральные измерения. [Описание наборов](../docs/ru/preprocessing.md).

In [ ]:
from preprocessing import reference_targets
from analysis.io import read_colors
target_folder = reference_targets.main(config, PROFILE)
target_tables = {p.stem: read_colors(p) for p in target_folder.glob("*.csv")}
reporting.targets(target_tables, LANGUAGE)

## 6. Измеренные спектры → пропускание и оптическая плотность

Для **обоих микроскопов** делим имеющиеся спектральные экспорты на записанную экспозицию; по измеренному осветителю рассчитываем T и OD. В записи упомянуты 10 кадров сигнала и 10 фона, но входы уже являются экспортом прибора. Недокументированное вычитание фона не придумывается и не повторяется.

Графики ниже показывают выбранные поля Meiji. Для сравнения охватов пропускание стекол сочетается с общим D50. Для калибровки каждый микроскоп сохраняет **свой измеренный осветитель**. [Обработка и ограничения съемки](../docs/ru/measured_fields.md).

In [ ]:
from preprocessing import measured_fields
measured_fields.main(config, PROFILE)
figures.plot_measured(config, PROFILE, LANGUAGE)
reporting.spectral_quality(config, PROFILE, LANGUAGE)

Неположительные значения поля/осветителя маскируются до логарифма. Измеренный масштаб OD стекла сохраняется. T>1 отмечается без скрытого ограничения: это может отражать различие режимов или неопределенность, которую код не устраняет. Сохраняемое соглашение интегрирования описано в руководстве по исправлениям.

## 7. Изображения → зарегистрированные цвета полей

Усредняем пиксели каждого пронумерованного кадра, затем средние кадров с одинаковым весом. Первый запуск читает 1290 изображений; последующие используют кэш. Спектральные эталоны и зарегистрированные Lab сопоставляются по точным именам. **Выдержка камеры не берется из таблицы экспозиций спектрометра.** [Обработка изображений](../docs/ru/images.md).

In [ ]:
from preprocessing import image_fields
image_fields.main(config, PROFILE)
registered = read_colors(config.artifacts / PROFILE / "meiji/registered_lab.csv")
reference = read_colors(config.artifacts / PROFILE / "meiji/reference_lab.csv")
import pandas as pd
pd.DataFrame({"L": registered.values[:6, 0], "a": registered.values[:6, 1],
              "b": registered.values[:6, 2]}, index=registered.names[:6])

## 8. Все наборы в xy и рисунок охватов

Интегрируем красители и пропускание стекол, читаем подготовленные меры, строим треугольник откликов CCD. **xy описывает только цветность, без яркости.** Цветной фон нужен для просмотра и не участвует в расчете покрытия.

Статический рисунок показывает поля стеклянной меры, ее оболочку, оболочки красителей и Kodak Q60, точку D50. В таблице **эталон — оболочка красителей**, как при сравнении покрытия в статье. Покрытие — площадь пересечения, деленная на площадь эталона; плотность — число полей, деленное на площадь собственной оболочки.

In [ ]:
from analysis.workflow import calculate_gamuts, coverage_table
datasets, color_tables = calculate_gamuts(config, PROFILE)
figures.plot_paper_gamut(config, datasets, PROFILE, LANGUAGE)
coverage = coverage_table(datasets, reference="dyes")
reporting.table(coverage, LANGUAGE)

Исправленная оболочка стекол покрывает **100%** этой оболочки красителей. Это следствие измененной трактовки OD, не доказательство более точной цветовой калибровки. Сравнивать следует с 99.6716% исторического расчета по доступным данным; опубликованные 99.4% не объявляются восстановленными.

## 9. Полиномиальная коррекция внутри каждого микроскопа

Обучаем **зарегистрированные Lab → спектральные эталонные Lab**, отдельно для Meiji и Olympus. Сравниваем степени 1–5 с перекрестными членами и без них. Исторический показатель и стандартное среднее ΔE76 различаются:

* Стандартное среднее: усреднение `sqrt(ΔL² + Δa² + Δb²)` по полям.
* Исторический показатель: `(sqrt(sum(ΔL²)) + sqrt(sum(Δa²)) + sqrt(sum(Δb²))) / N`.

Исторический показатель воспроизводит округленные **0.75 и 0.66**, но его нельзя называть средним ΔE76. [Эксперимент и определения](../docs/ru/calibration.md).

In [ ]:
from analysis.calibration import run_experiment
comparison, models, pairs = run_experiment(config, PROFILE)
figures.plot_calibration(config, comparison, PROFILE, LANGUAGE)
reporting.selected_models(comparison, LANGUAGE)

В исправленном обучении 97 полей, прямые спектральные Lab и нормированные входы с допущением sRGB. Центрирование и масштаб определяются по обучающим данным. Модель без перекрестных членов использует степени всех входных каналов согласно аддитивной формуле статьи.

Следующая таблица сравнивает **обучение и пятикратную оценку на отложенных полях**. Откладываются целые поля, не независимые сеансы съемки. Большие ошибки степени 5 сохранены. Для демонстрации изображения остаются степени 3/4 из статьи, чтобы не менять эксперимент молча.

In [ ]:
columns = ["system", "degree", "cross_terms", "mean_delta_e76", "cv_mean_delta_e76"]
reporting.table(comparison[columns], LANGUAGE)

## 10. Применить преобразование к изображению

Для каждого микроскопа применяем его модель ко всем пикселям зарегистрированного изображения стекла **СЗС-22**. Третья панель — равномерный цвет спектрального эталона для просмотра. Пространственное эталонное изображение здесь не измерялось.

Таблица различает ошибку среднего цвета поля и среднюю ошибку пикселей. ΔE использует **Lab без ограничения**, а RGB для просмотра ограничивается диапазоном дисплея. Обучение на средних цветах не является автоматической проверкой модели для каждого пикселя или окрашенной ткани.

In [ ]:
image_metrics = []
for system in ("meiji", "olympus"):
    metrics = figures.image_example(config, models[system], pairs[system],
                                    system, PROFILE, LANGUAGE)
    image_metrics.append(metrics)
reporting.save_image_metrics(config, PROFILE, image_metrics)
reporting.table(image_metrics, LANGUAGE)

## 11. Интерактивное сравнение наборов целиком

Переключайте **точки** и **оболочки** целых наборов. Выберите эталонную оболочку; по умолчанию это приближенный треугольник CCD. Таблица содержит видимые наборы и всегда эталон. Скрытие набора не меняет сохраненные координаты или определение оболочки.

Эта же диаграмма сохраняется в `artifacts/<вариант>/chromaticity_ru.html` и открывается в браузере без интернета. Кнопка **Сохранить SVG** экспортирует текущий выбор. Если выполнение сценариев блокнота заблокировано, разрешите доверие локально созданному блокноту или откройте HTML отдельно. Рисунки, коэффициенты, ошибки полей и версии библиотек сохраняются рядом в каталоге результатов.

In [ ]:
from analysis.interactive import create_explorer
reporting.save_environment(config, PROFILE)
create_explorer(config, datasets, PROFILE, LANGUAGE)